# CodiEsp - 1000 Clinical Cases - Knowledge Graph

Data responses were generated from GPT-4o using a prompt template which included the clinical cases: ICD-10 diagnoses codes and the clinical notes.  Each ICD code along with the clinical notes was used to determine the relations type between the ICD-10 titles (Casual, Risk Factors, Associations, or no direct relationship)  

<hr style="height:1rem;">

## 1.0 Imports

In [ ]:
import pandas as pd

import json
import os

import networkx as nx
import matplotlib.pyplot as plt

from pyvis.network import Network
import re
from collections import defaultdict

<hr style="height:1rem;">

## 2.0 Function Declarations

In [ ]:
def plot_relationship_graph(df, rel_type):
    """
    Generate and plot a graph for a specific relationship type between ICD-10 codes.

    Parameters:
    - df: DataFrame containing source and target ICD-10 codes with relationships.
    - rel_type: The relationship type to filter the DataFrame by (e.g., "Leads to").
    """
    # Filter the DataFrame to include only rows with the specified relationship type
    filtered_df = df[df['relationship_type'] == rel_type]

    # Build the graph
    G = nx.DiGraph()

    # Add edges for the filtered relationship type
    for _, row in filtered_df.iterrows():
        G.add_edge(
            row['source_icd10_code'],
            row['target_icd10_code'],
            source_title=row['SourceICDTitle'],
            target_title=row['DestinationICDTitle']
        )

    # Plot the graph
    plt.figure(figsize=(10, 6))
    pos = nx.spring_layout(G)
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=1000, font_size=8)
    plt.title(f"Graph for Relationship Type: {rel_type}")
    plt.show()

In [ ]:
def find_descendants_by_level(graph, source, level=1, max_depth=10):
    """
    Recursively find descendants for a source ICD-10 code, up to a max depth.
    - graph: The directed graph structure (e.g., a networkx graph) containing ICD-10 codes as nodes.
    - source: The ICD-10 code for which descendants need to be found.
    - level: The current level of descendants being explored (default is 1).
    - max_depth: The maximum depth up to which descendants should be explored (default is 10).
    """
    # Base case: stop the recursion if the current level exceeds max depth
    if level > max_depth:
        return {}

    # Initialize an empty dictionary to store the descendants found at the current level
    descendants = {}

    # Loop through each target ICD-10 code that is a successor of the current source
    for target in graph.successors(source):
        # Add each descendant to the dictionary, along with its relationship and level information
        descendants[target] = {
            "relationship": graph[source][target]['relationship'],  # Relationship between source and target ICD-10 codes
            "level": level  # Current level of descendants
        }

        # Recursively call the function to find descendants for the next level (target as the new source)
        descendants.update(find_descendants_by_level(graph, target, level + 1, max_depth))

    # Return the dictionary containing all descendants found up to the specified depth
    return descendants

In [ ]:
def plot_descendants_graph(df, source_code, max_depth=10):
    """
    Generate and plot a graph of descendants for a given source ICD-10 code, up to a maximum depth.

    Parameters:
    - df: DataFrame containing source and target ICD-10 codes with relationships.
    - source_code: The source ICD-10 code for which descendants are to be found.
    - max_depth: Maximum depth for finding descendants (default is 10).
    """
    # Filter the dataframe to get rows where the source_icd10_code matches the chosen source_code
    descendants_df = df[df['source_icd10_code'] == source_code]

    # Build the graph for descendants
    G = nx.DiGraph()

    # Add edges for immediate descendants
    for _, row in descendants_df.iterrows():
        G.add_edge(
            row['source_icd10_code'],
            row['target_icd10_code'],
            relationship=row['relationship_type'],
            target_title=row['DestinationICDTitle']
        )

    # Get all descendants for the source code, up to a maximum depth
    descendants_by_level = find_descendants_by_level(G, source_code, level=1, max_depth=max_depth)

    # Add the found descendants to the graph for visualization
    for descendant in descendants_by_level:
        if descendant not in G:
            G.add_node(descendant)

    # Plot the graph
    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(G, k=0.5)
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=1000, font_size=10, arrows=True)

    # Label the edges with relationships
    edge_labels = nx.get_edge_attributes(G, 'relationship')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)

    # Add title
    plt.title(f"Graph for Source ICD-10 Code: {source_code} and its Descendants (up to level {max_depth})")
    plt.show()

In [ ]:
def build_icd_pyvis_network(df, icd_codes, notebook=False):
    """
    Build a pyvis network graph using ICD codes and a DataFrame.

    Args:
        df (pd.DataFrame): DataFrame with columns including:
            - source_icd10_code, target_icd10_code
            - SourceICDTitle, DestinationICDTitle
            - relationship_type
        icd_codes (set): Set of ICD-10 codes to filter by.
        notebook (bool): Set True if running in a notebook (e.g., Jupyter).

    Returns:
        pyvis.network.Network: The network graph object.
    """
    net = Network(notebook=True, height="800px", width="100%", directed=True)
    net.force_atlas_2based()

    added_nodes = set()

    # Filter for rows where source or target ICD code is in the input set
    filtered_df = df[
        (df['source_icd10_code'].isin(icd_codes)) | 
        (df['target_icd10_code'].isin(icd_codes))
    ]

    for _, row in filtered_df.iterrows():
        source_code = row['source_icd10_code']
        target_code = row['target_icd10_code']

        source_label = f"{source_code}: {row['SourceICDTitle']}"
        target_label = f"{target_code}: {row['DestinationICDTitle']}"
        relation = row['relationship_type']

        # Add source node
        if source_code not in added_nodes:
            color = 'lightblue' if source_code in icd_codes else 'gray'
            net.add_node(source_code, label=source_label, title=row['source_name'], color=color)
            added_nodes.add(source_code)

        # Add target node
        if target_code not in added_nodes:
            color = 'lightblue' if target_code in icd_codes else 'gray'
            net.add_node(target_code, label=target_label, title=row['target_name'], color=color)
            added_nodes.add(target_code)

        # Add edge
        net.add_edge(source_code, target_code, label=relation)

    return net

In [ ]:
def build_icd_pyvis_network2(df, icd_codes, notebook=False):
    """
    Build a pyvis network graph for each ICD-10 code's 1-hop (1-tier) neighbors.

    Args:
        df (pd.DataFrame): DataFrame with columns:
            - source_icd10_code, target_icd10_code
            - SourceICDTitle, DestinationICDTitle
            - relationship_type
        icd_codes (set): Set of ICD-10 codes to build 1-tier networks for.
        notebook (bool): True if running in a Jupyter notebook.

    Returns:
        pyvis.network.Network: Pyvis Network with all 1-hop relationships.
    """
    net = Network(notebook=True, height="800px", width="100%", directed=True)
    net.force_atlas_2based()
    
    added_nodes = set()

    for icd in icd_codes:
        # Extract 1-hop rows for current ICD code
        sub_df = df[
            (df['source_icd10_code'] == icd) |
            (df['target_icd10_code'] == icd)
        ]

        for _, row in sub_df.iterrows():
            src_code = row['source_icd10_code']
            tgt_code = row['target_icd10_code']
            relation = row['relationship_type']

            src_label = f"{src_code}: {row['SourceICDTitle']}"
            tgt_label = f"{tgt_code}: {row['DestinationICDTitle']}"

            # Add source node
            if src_code not in added_nodes:
                color = 'lightblue' if src_code in icd_codes else 'lightgray'
                net.add_node(src_code, label=src_label, title=row['source_name'], color=color)
                added_nodes.add(src_code)

            # Add target node
            if tgt_code not in added_nodes:
                color = 'lightblue' if tgt_code in icd_codes else 'lightgray'
                net.add_node(tgt_code, label=tgt_label, title=row['target_name'], color=color)
                added_nodes.add(tgt_code)

            # Add edge
            net.add_edge(src_code, tgt_code, label=relation)

    return net

In [ ]:
def extract_1hop_source_icd_nodes_edges_no_is_a(df, icd_codes):
    """
    Extract 1-hop ICD-10 target nodes and edges where input ICD codes are sources,
    excluding 'Is a' relationships and removing duplicates.

    Args:
        df (pd.DataFrame): DataFrame with columns:
            - source_icd10_code
            - target_icd10_code
            - relationship_type
        icd_codes (set): ICD-10 codes to filter on as source only.

    Returns:
        nodes (set): Unique ICD-10 codes (source and target).
        edges (list): Unique edges as dicts with keys:
            'source', 'target', 'relationship', 'confidence', 'supporting_text'.
    """
    filtered_df = df[
        (df["source_icd10_code"].isin(icd_codes)) & 
        (df["relationship_type"].str.lower() != "is a")
    ]

    nodes = set()
    edges_set = set()

    for _, row in filtered_df.iterrows():
        src = row["source_icd10_code"]
        tgt = row["target_icd10_code"]
        rel = row["relationship_type"]

        nodes.add(src)
        nodes.add(tgt)

        # Use a tuple to store unique edge info in the set
        edge_tuple = (src, tgt, rel)
        edges_set.add(edge_tuple)

    # Convert tuples back to dicts with extra keys empty
    edges = [
        {
            "source": src,
            "target": tgt,
            "relationship": rel,
            "confidence": "",
            "supporting_text": ""
        }
        for src, tgt, rel in edges_set
    ]

    return nodes, edges

<hr style="height:1rem;">

## 3.0 Read Data into DataFrame - Ground-Truth

In [ ]:
snomed_ct_file = "../data/snomedct/snomed_ct_icd10_cm.csv"
df = pd.read_csv(snomed_ct_file, dtype=str)
print(len(df))

In [ ]:
df = df[[
    'SourceId', 
    'SourceName',
    'SourceICDCode',
    'SourceICDTitle',
    'DestinationId',
    'DestinationName',
    'DestinationICDCode',
    'DestinationICDTitle',
    'TypeName'
]]                                                                # Replace with your desired column order
df = df.rename(columns={
    'Id': 'id', 
    'SourceId': 'source_id',
    'SourceName': 'source_name',
    'SourceICDCode': 'source_icd10_code',
    #'SourceICDTitle': 'source_icd10_description',
    'DestinationId': 'target_id',
    'DestinationName': 'target_name',
    'DestinationICDCode': 'target_icd10_code',
    #'DestinationICDTitle': 'target_icd10_description',
    'TypeName': 'relationship_type'
})                                                                # Rename columns
df = df.sort_values(by='source_icd10_code')   # Sorts in ascending order
df = df.dropna(subset=['source_icd10_code', 'target_icd10_code'])  # remove nan
print(len(df))
df.head(10)

<hr style="height:1rem;">

## 5.0 Build Knowledge Graph

In [ ]:
node_dict = {}
for _, row in df.iterrows():
    sid = row['source_icd10_code']
    tid = row['target_icd10_code']

    if sid not in node_dict:
        node_dict[sid] = {
            "id": sid,
            "name": row.get("SourceICDTitle", "Unknown"),
            "type": "ICD"
        }
    if tid not in node_dict:
        node_dict[tid] = {
            "id": tid,
            "name": row.get("DestinationICDTitle", "Unknown"),
            "type": "ICD"
        }

# Build edges list
edges_list = []
for _, row in df.iterrows():
    edges_list.append({
        "source": row['source_icd10_code'],
        "target": row['target_icd10_code'],
        "relationship": row.get('relationship_type', 'unknown'),
        "confidence": row.get('confidence', 'unknown')
    })

#### Change clinical case

In [ ]:
USE_CLINICAL_CASE = 'A'
#USE_CLINICAL_CASE = 'B'
#USE_CLINICAL_CASE = 'C'
#USE_CLINICAL_CASE = 'D'
#USE_CLINICAL_CASE = 'E'
#USE_CLINICAL_CASE = 'F'

### Generated KG data from snomed_ct_kg_pd1 directory - copied here

In [ ]:
if USE_CLINICAL_CASE == 'A':
    clinical_case_name = 'clinical_case_a'
    ground_truth_icds = {
        "R52", 
        "K08.89"
    }
    predicted_icds = {
        "R52", 
        "K08.89"
    }
    predicted_edges_list = [
        {"source": "R52", "target": "K08.89", "relationship": "ASSOCIATED_WITH", "confidence": "high", "supporting_text": "The patient experienced pain in the upper right hemiarcade, which was linked to the observed fissure in the upper right premolar."}
    ]
elif USE_CLINICAL_CASE == 'B':
    clinical_case_name = 'clinical_case_b'
    ground_truth_icds = {
        "K08.109"
    }
    predicted_icds = {
        "K08.109", 
        "SNOMED_CT_127013003",
        "ICD10_K08.3",
    }
    predicted_edges_list = [
        {"source": "K08.109", "target": "SNOMED_CT_12701300", "relationship": "ASSOCIATED_WITH", "confidence": "High", "supporting_text": "The patient was diagnosed with partial edentulism, which is a related condition to the complete loss of teeth."},
        {"source": "SNOMED_CT_127013003", "target": "ICD10_K08.3", "relationship": "ASSOCIATED_WITH", "confidence": "Medium", "supporting_text": "Partial edentulism was mentioned along with severe alveolar sequestration of the edentulous premaxilla region."}
    ]
elif USE_CLINICAL_CASE == 'C':
    clinical_case_name = 'clinical_case_c'
    ground_truth_icds = {
        "E11.9", 
        "M79.81",
        "R05",
        "T14.8"
    }
    predicted_icds = {
        "M79.81", 
        "E11.9", 
        "R05",
        "T14.8"
    }
    predicted_edges_list = [
        {"source": "M79.81", "target": "T14.8", "relationship": "Complication of", "confidence": "High", "supporting_text": "A blow of cough followed by a bulging of the entire right anterior rectum, very painful and hard consistency."},
        {"source": "E11.9", "target": "M79.81", "relationship": "Associated with", "confidence": "Moderate", "supporting_text": "The patient has a history of IDDM (Insulin-dependent diabetes mellitus), which could contribute to the development of soft tissue hematoma."},
        {"source": "R05", "target": "T14.8", "relationship": "Cause of", "confidence": "High", "supporting_text": "The patient presented with a blow of cough, which led to the subsequent bulging of the right anterior rectum and the development of a hematoma."}
    ]
elif USE_CLINICAL_CASE == 'D':
    clinical_case_name = 'clinical_case_d'
    ground_truth_icds = {
        "B00.9", 
        "B99.9", 
        "C15.9", 
        "F17.210", 
        "K22.2", 
        "R13.10", 
        "R50.9", 
        "R60.9", 
        "R63.4"
    }
    predicted_icds = {
        "C15.9", 
        "K22.2",
        "R50.9",
        "B00.9", 
        "R13.10", 
        "F17.210", 
        "R60.9",
        "B99.9",
        "R63.4"
    }
    predicted_edges_list = [
        {"source": "C15.9", "target": "K22.2", "relationship": "COMPLICATES", "confidence": "High", "supporting_text": "Gastroscopy showed partial stenosis of the esophageal lumen, likely caused by malignant neoplasm."},
        {"source": "R13.10", "target": "K22.2", "relationship": "ASSOCIATED_WITH", "confidence": "High", "supporting_text": "Dysphagia to solids and liquids was mentioned, which is commonly associated with esophageal obstruction."},
        {"source": "R50.9", "target": "K22.2", "relationship": "ASSOCIATED_WITH", "confidence": "Moderate", "supporting_text": "Fever was noted occasionally, which can be associated with esophageal obstruction or infection."},
        {"source": "F17.210", "target": "C15.9", "relationship": "INCREASES_RISK", "confidence": "High", "supporting_text": "Active smoking is a well-known risk factor for developing esophageal cancer."},
        {"source": "R63.4", "target": "R13.10", "relationship": "ASSOCIATED_WITH", "confidence": "Low", "supporting_text": "The clinical notes did not mention significant weight loss, but it is often seen with dysphagia in various conditions."}
    ]    
elif USE_CLINICAL_CASE == 'E':
    clinical_case_name = 'clinical_case_e'
    ground_truth_icds = {
        "B99.9", 
        "C81.90", 
        "G93.40", 
        "K72.00", 
        "K72.90", 
        "K92.1", 
        "K92.2", 
        "R58", 
        "R69", 
        "R74.0"
    }
    predicted_icds = {
        "B99.9",
        "C81.90", 
        "R74.0", 
        "K72.90", 
        "R58", 
        "K92.1", 
        "K92.2", 
        "G93.40", 
        "K72.00", 
        "R69"
    }
    predicted_edges_list = [
        {"source": "C81.90", "target": "K72.00", "relationship": "COMPLICATES", "confidence": "High", "supporting_text": "The patient, undergoing chemotherapy for Hodgkin's disease, developed acute liver failure."},
        {"source": "C81.90", "target": "K92.1", "relationship": "COMPLICATES", "confidence": "High", "supporting_text": "During chemotherapy, the patient developed melena, a sign of gastrointestinal bleeding."},
        {"source": "K72.00", "target": "G93.40", "relationship": "CAUSES", "confidence": "High", "supporting_text": "The patient had signs of encephalopathy due to acute liver failure."},
        {"source": "K92.2", "target": "K72.00", "relationship": "CAUSES", "confidence": "High",  "supporting_text": "Massive gastrointestinal bleeding contributed to acute liver failure."},
        {"source": "B99.9", "target": "K72.00", "relationship": "INCREASES_RISK", "confidence": "Medium", "supporting_text": "Histological analysis suggested a herpetic infection with necrotizing inflammatory lesions."}
    ]
    
elif USE_CLINICAL_CASE == 'F':
    clinical_case_name = 'clinical_case_f'
    ground_truth_icds = {
        "I10",
        "K42.9",
        "K56.60",
        "K92.2",
        "N80.5",
        "R10.32",
        "R10.814",
        "R19.8",
        "R58"
    }
    predicted_icds = {
        "K92.2",
        "N80.5",
        "K56.60",
        "R19.8",
        "I10",
        "K42.9",
        "R58",
        "R10.814",
        "R10.32"
    }
    predicted_edges_list = [
        {"source": "K92.2", "target": "N80.5", "relationship": "COMPLICATES", "confidence": "High", "supporting_text": "Histological study confirmed colon endometriosis with transmural involvement causing areas of fibrosis and hemorrhage."},
        {"source": "N80.5", "target": "K56.60", "relationship": "COMPLICATES", "confidence": "High", "supporting_text": "The stenosing area was resected with end-to-end anastomosis, indicating an obstruction caused by endometriosis."},
        {"source": "K92.2", "target": "R58", "relationship": "ASSOCIATED_WITH", "confidence": "Moderate", "supporting_text": "The clinical notes mention hemorrhage related to gastrointestinal conditions."},
        {"source": "I10", "target": "K92.2", "relationship": "INCREASES_RISK", "confidence": "Low", "supporting_text": "The patient has a history of hypertension, which may complicate gastrointestinal hemorrhage."},
        {"source": "K42.9", "target": "N80.5", "relationship": "ASSOCIATED_WITH", "confidence": "Moderate", "supporting_text": "The patient has a history of umbilical hernia, which could be associated with abdominal pain and gastrointestinal symptoms."},
        {"source": "R10.814", "target": "R10.32", "relationship": "ASSOCIATED_WITH", "confidence": "High", "supporting_text": "The patient presented with abdominal tenderness and pain, both localized in the left lower quadrant."}
    ]    

In [ ]:
#net = build_icd_pyvis_network(df, ground_truth_icds)
#net.show("icd_graph.html")  # writes and opens a file

In [ ]:
#net = build_icd_pyvis_network2(df, ground_truth_icds)
#net.show("icd_graph.html")  # writes and opens a file

In [ ]:
ground_truth_nodes, ground_truth_edges = extract_1hop_source_icd_nodes_edges_no_is_a(df, ground_truth_icds)

In [ ]:
# === Settings ===
grid_spacing = 200  # Space between nodes
nodes_per_row = 5   # Adjust based on number of nodes

# === Union of all ICDs ===
all_icds = ground_truth_icds.union(predicted_icds)

# === Create PyVis network ===
sub_net = Network(notebook=True, height="1000px", width="100%", directed=True)
#sub_net.force_atlas_2based()
sub_net.set_options("""
{
  "physics": {
    "enabled": false
  },
  "layout": {
    "improvedLayout": false
  },
  "interaction": {
    "dragNodes": true,
    "dragView": false,
    "zoomView": false
  }
}
""")

added_nodes = set()
subgraph_nodes = []
subgraph_edges = []

# === Layout helper: assign fixed grid positions ===
icd_list = sorted(all_icds)
positions = {}
for idx, icd_code in enumerate(icd_list):
    row = idx // nodes_per_row
    col = idx % nodes_per_row
    x = col * grid_spacing
    y = row * grid_spacing
    positions[icd_code] = (x, y)

# === Add nodes with color logic ===
for icd_code in icd_list:
    x, y = positions[icd_code]
    if icd_code in node_dict:
        node_data = node_dict[icd_code]
        if icd_code in ground_truth_icds and icd_code in predicted_icds:
            color = "green"
        elif icd_code in predicted_icds:
            color = "orange"
        else:
            color = "lightblue"

        sub_net.add_node(icd_code,
                         label=f'{icd_code} ({node_data["name"]})',
                         title=node_data["type"],
                         color=color,
                         x=x,
                         y=y,
                         #fixed=True,
                         #physics=False
                        )
        added_nodes.add(icd_code)
        subgraph_nodes.append({
            "id": icd_code,
            "name": node_data["name"],
            "type": node_data["type"],
            "color": color,
            "x": x,
            "y": y
        })
    else:
        # Handle missing nodes
        if icd_code in predicted_icds:
            sub_net.add_node(icd_code,
                             label=f"{icd_code} (Not in SNOMED-CT)",
                             title="Predicted Only",
                             color="orange",
                             x=x,
                             y=y,
                             #fixed=True,
                             #physics=False
                            )
            added_nodes.add(icd_code)
            subgraph_nodes.append({
                "id": icd_code,
                "name": "Missing",
                "type": "Predicted Only",
                "color": "orange",
                "x": x,
                "y": y
            })

# === Edge sets ===
ground_truth_edges = {(e["source"], e["target"]) for e in edges_list}
predicted_edges = {(e["source"], e["target"]) for e in predicted_edges_list}
all_edges = ground_truth_edges.union(predicted_edges)

# === Helper: check if reversed edge exists ===
def is_bidirectional(src, tgt, edge_set):
    return (tgt, src) in edge_set

# === Add edges with curvature for bidirectional and edge labels ===
for (source, target) in all_edges:
    if source in added_nodes and target in added_nodes:
        # Lookup edge info from ground_truth or predicted edges
        edge_data = next((e for e in edges_list if e["source"] == source and e["target"] == target), None)
        if not edge_data:
            edge_data = next((e for e in predicted_edges_list if e["source"] == source and e["target"] == target), None)

        # Color logic
        if (source, target) in ground_truth_edges and (source, target) in predicted_edges:
            edge_color = "green"
        elif (source, target) in predicted_edges:
            edge_color = "orange"
        else:
            edge_color = "lightblue"

        # Curved edges for bidirectional
        if is_bidirectional(source, target, all_edges) and source != target:
            smooth_style = {"type": "curvedCW", "roundness": 0.3 } if source < target else {"type": "curvedCCW"}
        else:
            smooth_style = {"enabled": False}

        edge_label = edge_data["relationship"]
        if edge_color == 'orange':
            edge_label += ' (Not in SNOMED-CT)'

        # Add edge with label
        sub_net.add_edge(
            source,
            target,
            label=edge_label,
            title=f'{edge_data["relationship"]} ({edge_data.get("confidence", "N/A")})',
            color=edge_color,
            smooth=smooth_style,
            #length=25  # default is 100; smaller = shorter edge
        )

        subgraph_edges.append({
            "source": source,
            "target": target,
            "relationship": edge_data["relationship"],
            "confidence": edge_data.get("confidence", None),
            "color": edge_color
        })

# === Save outputs ===
safe_name = re.sub(r'\W+', '_', "_".join(sorted(all_icds)))
filename = f"{clinical_case_name}_snomed_ct_kg_subgraph_{safe_name}.html"
sub_net.show(filename)
print(f"Graph saved as '{filename}'.")

# Save CSVs
os.makedirs("./snomed_ct_kg_pd1", exist_ok=True)
pd.DataFrame(subgraph_nodes).to_csv(f"./snomed_ct_kg_pd1/{safe_name}_nodes.csv", index=False)
pd.DataFrame(subgraph_edges).to_csv(f"./snomed_ct_kg_pd1/{safe_name}_edges.csv", index=False)